<a href="https://colab.research.google.com/github/HSA2005-cloud/BIS-Lab/blob/main/1.%20Genetic%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import random

# Electricity Load Scheduling using Genetic Algorithm

POP_SIZE = 100
GENERATIONS = 300
MUTATION_RATE = 0.10
DAYS_PER_MONTH = 30

# Sample data for six time slots.
# Demand and supply are measured in kWh per time slot.
# Format: time, residential demand, industrial demand,
# supply, essential demand, cost per kWh, grid capacity

data = [
    ("12 AM - 4 AM", 20, 50, 70, 20, 4, 70),
    ("4 AM - 8 AM",  40, 60, 90, 30, 5, 90),
    ("8 AM - 12 PM", 70, 90, 120, 50, 7, 120),
    ("12 PM - 4 PM", 60, 100, 130, 50, 8, 130),
    ("4 PM - 8 PM", 100, 70, 140, 60, 10, 140),
    ("8 PM - 12 AM", 80, 40, 100, 40, 6, 100)
]


# Step 1: Generate a chromosome
# Each chromosome contains residential and industrial allocations.

def create_chromosome():
    residential = []
    industrial = []

    for slot in data:
        residential.append(random.uniform(0, slot[1]))
        industrial.append(random.uniform(0, slot[2]))

    chromosome = [residential, industrial]
    repair(chromosome)
    return chromosome


# Ensure supply and capacity limits are not exceeded
def repair(chromosome):
    residential, industrial = chromosome

    for i, slot in enumerate(data):
        rd, ind_demand = slot[1], slot[2]
        supply, capacity = slot[3], slot[6]

        residential[i] = max(0, min(residential[i], rd))
        industrial[i] = max(0, min(industrial[i], ind_demand))

        limit = min(supply, capacity)
        total = residential[i] + industrial[i]

        if total > limit:
            factor = limit / total
            residential[i] *= factor
            industrial[i] *= factor


# Step 2: Calculate fitness
# Lower fitness means a better schedule.
def fitness(chromosome):
    repair(chromosome)

    residential, industrial = chromosome
    total_cost = 0
    penalty = 0

    for i, slot in enumerate(data):
        rd, ind_demand = slot[1], slot[2]
        essential, cost = slot[4], slot[5]

        r = residential[i]
        ind = industrial[i]
        total = r + ind

        total_cost += total * cost

        # Give priority to essential electricity demand
        if total < essential:
            penalty += (essential - total) * 10000

        # Penalize unmet demand in both areas
        penalty += (rd - r) * 100
        penalty += (ind_demand - ind) * 80

    return total_cost + penalty


# Step 3: Select parents
def select_parent(population):
    candidates = random.sample(population, 5)
    return min(candidates, key=fitness)


# Step 4: Crossover
def crossover(parent1, parent2):
    child_r = []
    child_i = []

    for i in range(len(data)):
        if random.random() < 0.5:
            child_r.append(parent1[0][i])
            child_i.append(parent1[1][i])
        else:
            child_r.append(parent2[0][i])
            child_i.append(parent2[1][i])

    child = [child_r, child_i]
    repair(child)
    return child


# Step 5: Mutation
def mutate(chromosome):
    residential, industrial = chromosome

    for i, slot in enumerate(data):
        if random.random() < MUTATION_RATE:
            residential[i] = random.uniform(0, slot[1])

        if random.random() < MUTATION_RATE:
            industrial[i] = random.uniform(0, slot[2])

    repair(chromosome)


# Steps 1-7: Run the genetic algorithm
def genetic_algorithm():
    population = [
        create_chromosome() for _ in range(POP_SIZE)
    ]

    for generation in range(GENERATIONS):
        population.sort(key=fitness)

        # Preserve the best solution
        best = [
            population[0][0][:],
            population[0][1][:]
        ]

        new_population = [best]

        while len(new_population) < POP_SIZE:
            parent1 = select_parent(population)
            parent2 = select_parent(population)

            child = crossover(parent1, parent2)
            mutate(child)

            new_population.append(child)

        population = new_population

    population.sort(key=fitness)

    return [
        population[0][0][:],
        population[0][1][:]
    ]


# Step 8: Display the schedule and costs
def display_results(best):
    residential, industrial = best

    print("\nELECTRICITY LOAD SCHEDULING")
    print("=" * 92)

    print(
        f"{'Time':<16}"
        f"{'Residential':>14}"
        f"{'Industrial':>14}"
        f"{'Total Power':>14}"
        f"{'Cost':>14}"
    )

    print("-" * 92)

    daily_cost = 0

    for i, slot in enumerate(data):
        r = residential[i]
        ind = industrial[i]
        total = r + ind
        cost = total * slot[5]

        daily_cost += cost

        print(
            f"{slot[0]:<16}"
            f"{r:>14.2f}"
            f"{ind:>14.2f}"
            f"{total:>14.2f}"
            f"{cost:>14.2f}"
        )

    monthly_cost = daily_cost * DAYS_PER_MONTH

    print("-" * 92)
    print(f"Daily electricity cost   : {daily_cost:.2f}")
    print(f"Monthly electricity cost : {monthly_cost:.2f}")
    print(f"Monthly period           : {DAYS_PER_MONTH} days")

    # Find maximum and minimum allocations
    r_max = max(range(len(data)), key=lambda i: residential[i])
    r_min = min(range(len(data)), key=lambda i: residential[i])

    i_max = max(range(len(data)), key=lambda i: industrial[i])
    i_min = min(range(len(data)), key=lambda i: industrial[i])

    print("\nRESIDENTIAL AREA")
    print("-" * 40)
    print(
        f"More power : {data[r_max][0]} "
        f"({residential[r_max]:.2f} kWh)"
    )
    print(
        f"Less power : {data[r_min][0]} "
        f"({residential[r_min]:.2f} kWh)"
    )

    print("\nINDUSTRIAL AREA")
    print("-" * 40)
    print(
        f"More power : {data[i_max][0]} "
        f"({industrial[i_max]:.2f} kWh)"
    )
    print(
        f"Less power : {data[i_min][0]} "
        f"({industrial[i_min]:.2f} kWh)"
    )

    print("\nDEMAND SATISFACTION")
    print("-" * 40)

    total_r_demand = sum(slot[1] for slot in data)
    total_i_demand = sum(slot[2] for slot in data)
    total_r_given = sum(residential)
    total_i_given = sum(industrial)

    print(
        f"Residential demand supplied: "
        f"{total_r_given:.2f}/{total_r_demand:.2f} kWh "
        f"({100 * total_r_given / total_r_demand:.1f}%)"
    )

    print(
        f"Industrial demand supplied: "
        f"{total_i_given:.2f}/{total_i_demand:.2f} kWh "
        f"({100 * total_i_given / total_i_demand:.1f}%)"
    )


# Main program
if __name__ == "__main__":
    random.seed(42)  # Reproducible example
    best_schedule = genetic_algorithm()
    display_results(best_schedule)



ELECTRICITY LOAD SCHEDULING
Time               Residential    Industrial   Total Power          Cost
--------------------------------------------------------------------------------------------
12 AM - 4 AM             19.89         50.00         69.89        279.56
4 AM - 8 AM              39.80         50.20         90.00        450.00
8 AM - 12 PM             69.86         50.14        120.00        840.00
12 PM - 4 PM             59.77         70.23        130.00       1040.00
4 PM - 8 PM              99.04         40.96        140.00       1400.00
8 PM - 12 AM             78.89         21.11        100.00        600.00
--------------------------------------------------------------------------------------------
Daily electricity cost   : 4609.56
Monthly electricity cost : 138286.84
Monthly period           : 30 days

RESIDENTIAL AREA
----------------------------------------
More power : 4 PM - 8 PM (99.04 kWh)
Less power : 12 AM - 4 AM (19.89 kWh)

INDUSTRIAL AREA
----------------